# 1.0 — Visualização de embeddings (UMAP)

**Objetivo:** projetar embeddings de **indivíduos** (`embeddings_avg`) e **catálogo** (`embedding_gpt`) do mesmo parceiro em 2D (UMAP) e comparar clusters partner vs product.

**Ajuste antes de rodar:** `PARTNER`, `DATABASE_INDIVIDUALS` e `DATABASE_CATALOGS` (sufixo `v*` alinhado ao `data_version` em `wokibi_data.config.partner_parameters`).

**Dependências:** Mongo com embeddings materializados; `poetry install` + kernel do `.venv`. Ver [notebooks/README.md](../README.md).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from wokibi_data.service.mongo import MongoInterface
from wokibi_eval.evaluation.embedding import plot_umap, get_umap

In [ ]:
PARTNER = "nefrologia"

DATABASE_INDIVIDUALS = "wokibi_individuals_v20260830"
DATABASE_CATALOGS = "wokibi_catalogs_v20260830"

In [ ]:
mongo_partners = MongoInterface(DATABASE_INDIVIDUALS)
mongo_partners.set_collection(PARTNER)
mongo_items = MongoInterface(DATABASE_CATALOGS)
mongo_items.set_collection(PARTNER)

In [ ]:
embeddings_partners, labels_partners = mongo_partners.get_embeddings_array(
    embedding_field='embeddings_avg', label_field='nome', break_on_error=True)
embeddings_items, labels_items = mongo_items.get_embeddings_array(
    embedding_field='embedding_gpt', label_field='nome', break_on_error=True)

embeddings_array = np.concatenate([embeddings_partners, embeddings_items])
labels = np.concatenate([labels_partners, labels_items])

len(embeddings_partners), len(embeddings_items), len(embeddings_array)

In [ ]:
labels = []
for l in labels_partners:
    labels.append("partner")
for l in labels_items:
    labels.append("product")
len(labels)

In [ ]:
df = get_umap(embeddings_array, labels)
plot_umap(df, with_labels=True, title="Wokibi - Visualização de Embeddings com UMAP")

In [ ]:
df.shape

In [ ]:
df.sample(5)

In [ ]:
df.query("x < 6 and y > 7.5")

In [ ]:
df.query("y < 0")